# Split and move the DPR processing flow

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-672

See the associated:

  * Python module: [split_processor_flow.py](./split_processor_flow.py)
  * YAML file: [split_processor_flow.yaml](./split_processor_flow.yaml)

## 1. Initialisation

In [34]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [35]:
USE_DPR_MOCKUP = True

# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=2, use_mockup = USE_DPR_MOCKUP)
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)
display(dask_cluster_staging)

Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-eopf-mockup': http://dask-eopf-mockup:8000 ...
image = 50746b5e51c74a1f95287dfe45378533
Get existing dask cluster: '50746b5e51c74a1f95287dfe45378533'
Dask dashboard for 'dask-eopf-mockup': http://localhost:8703/clusters/50746b5e51c74a1f95287dfe45378533/status
Dask workers for 'dask-eopf-mockup' are up: 2/2
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
image = cde34781d56845f3ad67377dae29b15c
Get existing dask cluster: 'cde34781d56845f3ad67377dae29b15c'
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/cde34781d56845f3ad67377dae29b15c/status
Dask workers for 'dask-staging' are up: 2/2


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+----------+
| Package     | Client   | Scheduler | Workers  |
+-------------+----------+-----------+----------+
| dask        | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| distributed | 2024.5.2 | 2025.2.0  | 2025.2.0 |
| numpy       | 2.2.5    | 2.2.4     | 2.2.4    |
| tornado     | 6.3.3    | 6.4.2     | 6.4.2    |
+-------------+----------+-----------+----------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.5  | 2.2.4     | 2.2.4   |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module

In [36]:
# Create a test collection
TEST_COLLECTION_NAME = "SPLIT_FLOWS_TEST_COLLECTION"
collection = create_test_collection(TEST_COLLECTION_NAME)

# Check that it is empty
items = catalog_client.get_items(TEST_COLLECTION_NAME)
assert not list(items)

#CADIP_SESSION_FILTER = "id=S3A_20250109134406046340" # Session id "platform='sentinel-1a'" "id=S1A_20200105072204051312" S3A_20250109134406046340 | S1A_20200105072204051312
CADIP_SESSION_FILTER ="id=S1A_20200105072204051312"

14:54:02.189 [INFO] (rs_client.rs_client) Retrieving all items from collection 'jgaucher:SPLIT_FLOWS_TEST_COLLECTION'.


In [37]:
# Other imports
import os
import os.path as osp
from rs_common import prefect_utils
from rs_common.prefect_utils import *

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

flow_parameters = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_l0_demo_payload_dpr_mockup_template.yaml",
    "output_data_dir": f"{s3_output}/s3",
    "owner_id": OWNER_ID,
    "collection_name": TEST_COLLECTION_NAME,
    "cadip_stac_filter": CADIP_SESSION_FILTER,
    "staging_timeout": 120,
    "use_dpr_mockup": USE_DPR_MOCKUP,
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

14:54:02.471 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/logging_config.yaml'.

14:54:02.472 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml'.

14:54:02.473 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_3A.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_3A.yaml'.

14:54:02.473 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_dpr_mockup.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'.

14:54:02.490 | INFO    | prefect.S3Bucket - Uploaded 4 files from 'l0/config' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [38]:
from rs_workflows import on_demand_processing
flow_module = on_demand_processing.__file__

In [39]:
%%bash -s "$flow_module"
ln -sf $1

In [42]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory and resources contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)
await share_bucket.put_directory(local_path = "../../resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/code'


In [43]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./split_processor_flow.yaml"

╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'On-demand processing/On-demand processing' successfully created  │
│ with id 'a30ee720-4c3b-460c-99b2-53828fc0f447'.                              │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/a30ee720-4c3b-460c-99b2-53828fc0f447


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'On-demand processing/On-demand processing'



In [44]:
deploy_name = "mytest/sprint23-s3l0-demo-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint23-s3l0-demo-processor' ...
Wait for deployment of prefect flow: 'mytest/sprint2

ObjectNotFound: None

## 3. Run Prefect flow

In [36]:
output_data_dir = flow_parameters["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters) # flow parameters

Remove existing zarr products from: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s3'


In [37]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 'mytest/sprint23-s3l0-demo-processor'...
Created flow run 'wise-chihuahua'.
└── UUID: ed19b4ad-5552-4767-9662-93b9be384ca1
└── Parameters: {'input_config_dir': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/config', 'payload_file': 's3/s3_l0_demo_payload_dpr_mockup_template.yaml', 'output_data_dir': 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/l0/output/s3', 'owner_id': 'jgaucher', 'collection_name': 'SPLIT_FLOWS_TEST_COLLECTION', 'cadip_stac_filter': 'id=S1A_20200105072204051312', 'staging_timeout': 120, 'use_dpr_mockup': True}
└── Job Variables: {}
└── Scheduled start time: 2025-05-23 13:09:00 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/ed19b4ad-5552-4767-9662-93b9be384ca1
Watching flow run 'wise-chihuahua'...


13:09:05.470 | INFO    | prefect - Flow run is in state 'Pending'
13:09:08.778 | INFO    | prefect - Flow run is in state 'Running'
13:09:08.796 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


In [ ]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)
eopf_prod_ids = ["S03MWRL0__20221101T092439_6037_A307_T677", "S03OLCL0__20210629T044945_0119_A247_T219"]
for id in eopf_prod_ids:
    assert catalog_client.get_item(TEST_COLLECTION_NAME, id) 
   

## 6. Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)
    dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [45]:
from importlib import reload
debug_flow = True

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_eopf(scale=2)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *

In [48]:
if debug_flow:

    import sys
    from rs_workflows import on_demand_processing

    from rs_workflows.on_demand_processing import CadipFlowParams

    # Reload the flow and all rs-client-libraries modules
    reload(on_demand_processing)
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    results = await on_demand_processing.on_demand_processing(
        "aaa", "bbb", "ccc"
    )
    display(results)

15:01:28.644 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/84b55665-fb4b-4f70-8a34-62e525674b21

15:01:28.678 | INFO    | Flow run 'rousing-dodo' - Beginning flow run 'rousing-dodo' for flow 'On-demand processing'

15:01:28.681 | INFO    | Flow run 'rousing-dodo' - View at http://prefect-server:4200/runs/flow-run/84b55665-fb4b-4f70-8a34-62e525674b21

15:01:28.728 | CRITICAL | Task run 'myrun-5cd' - aaa

15:01:28.729 | CRITICAL | Task run 'myrun-5cd' - bbb

15:01:28.731 | CRITICAL | Task run 'myrun-5cd' - ccc

15:01:28.735 | INFO    | Task run 'myrun-5cd' - Finished in state Completed()

15:01:28.772 | INFO    | Flow run 'rousing-dodo' - Finished in state Completed()

None